In [0]:
from pyspark.sql import functions as F

CATALOG, SCHEMA = "payment_lakehouse", "pipeline"
LANDING    = f"/Volumes/{CATALOG}/{SCHEMA}/landing"
CHECKPOINT = f"{LANDING}/_checkpoints"
SCHEMAS    = f"{LANDING}/_schemas"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [0]:
# Auto Loader. Tracks processed files in RocksDB at the checkpoint, so it
# never relists the directory and each file is read exactly once.
# schemaEvolutionMode=rescue puts anything that doesn't fit the inferred
# schema into _rescued_data instead of dropping it.
raw = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"{SCHEMAS}/bronze")
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.inferColumnTypes", "true")
    .load(f"{LANDING}/events"))

# Provenance. ingest_ts is when WE saw it; event_ts is when it HAPPENED.
bronze = (raw
    .withColumn("ingest_ts",   F.current_timestamp())
    .withColumn("ingest_dt",   F.to_date(F.current_timestamp()))
    .withColumn("source_file", F.col("_metadata.file_path")))

In [0]:
# availableNow: processes all available data in rate-limited batches, then
# stops. Batch economics, streaming semantics — checkpointed and exactly-once.
# Trigger.Once is deprecated because it ignored maxFilesPerTrigger.
(bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{CHECKPOINT}/bronze")
    .option("mergeSchema", "true")
    .partitionBy("ingest_dt")
    .trigger(availableNow=True)
    .toTable(f"{CATALOG}.{SCHEMA}.bronze_events")
    .awaitTermination())

print("bronze loaded")

In [0]:
display(spark.sql("""
  SELECT count(*) AS rows,
         count(DISTINCT event_id) AS unique_events,
         count(*) - count(DISTINCT event_id) AS duplicates,
         sum(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rescued
  FROM bronze_events
"""))

In [0]:
display(spark.sql("SELECT * FROM bronze_events LIMIT 5"))
display(spark.sql("DESCRIBE HISTORY bronze_events"))

In [0]:
# Production variant — the only change is the source.
# kafka = (spark.readStream.format("kafka")
#     .option("kafka.bootstrap.servers", "broker:9092")
#     .option("subscribe", "payment-events")
#     .option("startingOffsets", "earliest")
#     .load()
#     .select(F.from_json(F.col("value").cast("string"), EVENT_SCHEMA).alias("e"),
#             F.col("timestamp").alias("kafka_ts"))
#     .select("e.*", "kafka_ts"))